In [85]:
import numpy as np
import pandas as pd
import timeit
import copy
import time
import itertools
import os
from joblib import Parallel, delayed
import json
import networkx as nx
from functions import *

In [1]:
import numpy as np
import pandas as pd
import timeit

In [2]:
import sys

sys.path.insert(
    0,
    "/Users/ryanli/Documents/Codex/2026-06-10/"
    "files-mentioned-by-the-user-functions1/outputs"
)


In [38]:

from dlx_experiment import solve
for z in [4,8,12]:#4:Realistic, 8:Random, 12:Clustered
    for INS in [500,1000,2500,5000]:#Number of users
        for h in range(20):
            
            output = solve(
                data_dir="/Users/ryanli/Documents/GitHub/DLX_plus/Data",
                instance=z,
                users=INS,
                sample=h,
                threads=10,
            )


            rows_color = output["rows_color"]
            orig_row = output["orig_row"]
            pos = output["pos"]
            D_color = output["D_color"]
            u = output["u"]
            N_max = output["N_max"]
            adj = output["adj"]
            back = output["back"]
    
            H=output["best_integer"]

            #print(output["best_value"])
            #print(output["best_time"])
            #print(output["best_backtrack_depth"])
            #print(output["best_percentile_index"])
            #print(output["best_integer"])
            pairs_file = os.path.join('Data/Beam Candidate/', 'Pairs_'+str(z)+'_'+str(INS)+'_sample'+str(h)+'.json')
            with open(pairs_file, 'r') as f:
                pairs = json.load(f)
            f = open('Data/Beam Candidate/Allocate_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.txt','r')
            allocate_data=f.read()
            f.close()
            allocate=eval(allocate_data)
            colour={}
            node={}
            reverse_pos = {v: k for k, v in pos.items()}
            for i in H:
                colour[reverse_pos[orig_row[i][0]]]=orig_row[i][1]
            for i,j in pairs:
                if i in colour and j in colour:
                    if colour[i]==colour[j]:
                        print("Break_colour",i,j)
            users=[]
            for i in colour:
                users.extend(allocate[i])
            if len(users)!=len(set(users)):
                print("Break_users")

In [91]:
def run_single_i(i, rows_color, D_color, orig_row, back, LB, LB_integer,N_max, u, n_colour):
    """
    Run DLX+ for one starting index i across all percentile settings.
    Returns local time/result dictionaries.
    """

    begin = i * n_colour

    # Slice once only
    rows_i = rows_color[begin:]
    orig_i = orig_row[begin:]
    D_i = D_color[begin:]

    backtrack_depth=None

    for p, node0 in enumerate(back):
        node = node0 - begin

        if node > 0:
            solver = DLX(
                u + 1,
                rows_i,
                orig_i,
                D_i,
                LB,
                N_max,
                node
            )

            for int_sol, obj in solver.search_range():
                if obj > LB:
                    LB = obj
                    LB_integer=[row + begin for row in int_sol.copy()]
                    backtrack_depth=p
            

    return LB,LB_integer,backtrack_depth

In [81]:
def cumulative_analysis(max_i, records, percentiles):
    """
    Analyse cumulative results up to index max_i.

    For each percentile, keep:
    - maximum running time across threads,
    - maximum objective value across threads.
    """

    per_p = {
        p: {"time": 0.0, "value": 0}
        for p in percentiles
    }

    for i, (time_list, value_list) in enumerate(records):
        if i > max_i:
            break

        for p in percentiles:
            if time_list[p] > per_p[p]["time"]:
                per_p[p]["time"] = time_list[p]

            if value_list[p] > per_p[p]["value"]:
                per_p[p]["value"] = value_list[p]

    aggregated = [
        {"p": p, "time": v["time"], "value": v["value"]}
        for p, v in per_p.items()
        if v["time"] > 0
    ]

    if not aggregated:
        return None, None

    best_value = max(r["value"] for r in aggregated)
    fastest_of_best = min(
        (r for r in aggregated if r["value"] == best_value),
        key=lambda r: r["time"]
    )

    return fastest_of_best

In [73]:
def run_single_i(i, rows_color, D_color, orig_row, back, LB, N_max, u, t, n_colour=4):
    """
    Run DLX+ for one starting index i across all percentile settings.
    Returns local time/result dictionaries.
    """

    begin = i * n_colour

    # Slice once only
    rows_i = rows_color[begin:]
    orig_i = orig_row[begin:]
    D_i = D_color[begin:]

    local_time = [0.0] * len(back)
    local_result = [LB] * len(back)

    best = LB

    for p, node0 in enumerate(back):
        node = node0 - begin

        t_start = timeit.default_timer()

        if node > 0:
            solver = DLX(
                u + 1,
                rows_i,
                orig_i,
                D_i,
                LB,
                N_max,
                node
            )

            for _, obj in solver.search_range():
                if obj > best:
                    best = obj

        local_time[p] = t + (timeit.default_timer() - t_start)
        local_result[p] = best

    return local_time, local_result

In [4]:


percentiles = [0.01,0.05, 0.25, 0.50, 0.75, 0.95, 0.99]#Backtracking depth percentile based on the beam load distribution
thread = 10# using 10 threads
dfs_best_of_fastest = []
dfs_fastest_of_best = []
N_max=60
n_colour=4
R=range(n_colour)
Gurobi_SOL=pd.DataFrame(columns=['Instance','User','Simulation','Solver','Computation Time(s)','Total Allocated Demand'])
OR_TOOLS_SOL=pd.DataFrame(columns=['Instance','User','Simulation','Solver','Computation Time(s)','Total Allocated Demand'])
for z in [4,8,12]:#4:Realistic, 8:Random, 12:Clustered
    for INS in [500,1000,2500,5000]:#Number of users
        Data=pd.read_excel('Data/User Data/Instance_'+str(z)+'.xlsx')
        data=Data.iloc[:INS].copy()
        for h in range(20): #problems     
            beams=pd.read_csv('Data/Beam Candidate/Beams_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.csv',index_col=0)
            f = open('Data/Beam Candidate/Allocate_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.txt','r')
            allocate_data=f.read()
            f.close()
            allocate=eval(allocate_data)
            pairs_file = os.path.join('Data/Beam Candidate/', 'Pairs_'+str(z)+'_'+str(INS)+'_sample'+str(h)+'.json')
            with open(pairs_file, 'r') as f:
                pairs = json.load(f)

            rows_color, orig_row, pos, D_color,u,adj,cum_demands,back=input_structure(beams,allocate,data,pairs,percentiles)

            Cons=incidence_transpose_fast(rows_color)

            t_start = timeit.default_timer()

            guro_integer,guro_obj,guro_solver_time=gurobi_SPK(rows_color,D_color,N_max,Cons)

            t_end = timeit.default_timer()

            Gurobi_SOL.loc[len(Gurobi_SOL)]=[z,INS,h,'Gurobi',t_end-t_start,guro_obj]

            t_start = timeit.default_timer()

            or_integer,or_obj,or_solver_time=cpsat_SPK(rows_color,D_color,N_max,Cons, log=True)

            t_end = timeit.default_timer()

            OR_TOOLS_SOL.loc[len(OR_TOOLS_SOL)]=[z,INS,h,'OR_Tools',t_end-t_start,or_obj]

            t_start = timeit.default_timer()

            LB, greedy_sol= greedy_solution(rows_color, orig_row, pos, D_color, N_max, adj)

            t_end = timeit.default_timer()
            t=t_end-t_start

            #Testing on various backtracking depths
       

            results = Parallel(n_jobs=thread,backend="loky")(delayed(run_single_i)(i,rows_color,D_color,orig_row,back,LB,N_max,u,t)
                               for i in range(thread)) 
            
            best_of_fastest, fastest_of_best = cumulative_analysis(thread-1, results, range(len(back)))
            # ---------- Best of Fastest ----------
            dfs_best_of_fastest.append({
                "Instance": z,
                "User": INS,
                "Simulation": h,
                "Backtracking based on percentiles": best_of_fastest["p"],
                "Total Allocated Demand": best_of_fastest["value"],
                "Computation Time(s)": best_of_fastest["time"],
            })

            # ---------- Fastest of Best ----------
            dfs_fastest_of_best.append({
                "Instance": z,
                "User": INS,
                "Simulation": h,
                "Backtracking percentiles idx": fastest_of_best["p"],
                "Total Allocated Demand": fastest_of_best["value"],
                "Computation Time(s)": fastest_of_best["time"],               
            })

            print(f"Completed: z={z}, INS={INS}, simulation={h}")


# ---------------- Convert to DataFrames ----------------

DLX_v1 = pd.DataFrame(dfs_fastest_of_best)
DLX_v2 = pd.DataFrame(dfs_best_of_fastest)
DLX_v1.to_csv('DLXv1_results.csv')
DLX_v2.to_csv('DLXv2_results.csv')
Gurobi_SOL.to_csv('Gurobi_results.csv')
OR_TOOLS_SOL.to_csv('OR_TOOLS_results.csv')

Set parameter Username
Academic license - for non-commercial use only - expires 2027-04-22
Set parameter TimeLimit to value 3600
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (mac64[rosetta2] - Darwin 23.6.0 23G93)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 1853 rows, 1188 columns and 14996 nonzeros
Model fingerprint: 0x46f38d97
Variable types: 0 continuous, 1188 integer (1188 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e+02, 1e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 6e+01]
Found heuristic solution: objective 950350.00000
Presolve removed 813 rows and 106 columns
Presolve time: 0.03s
Presolved: 1040 rows, 1082 columns, 13305 nonzeros
Found heuristic solution: objective 950350.00000
Variable types: 0 continuous, 1082 integer (1081 binary)

Root relaxation: objective 9.796917e+05, 325 iterations, 0.00 seconds (0.00 work units)

    No

In [86]:
            t_start = timeit.default_timer()

            LB, greedy_sol= greedy_solution(rows_color, orig_row, pos, D_color, N_max, adj)

            t_end = timeit.default_timer()
            t=t_end-t_start

            #Testing on various backtracking depths
       

            results = Parallel(n_jobs=thread,backend="loky")(delayed(run_single_i)(i,rows_color,D_color,orig_row,back,LB,N_max,u,t)
                               for i in range(thread)) 

TypeError: run_single_i() missing 2 required positional arguments: 'thread_time' and 'thread_result'

In [87]:
            LB, greedy_sol= greedy_solution(rows_color, orig_row, pos, D_color, N_max, adj)

            t_end = timeit.default_timer()
            t=t_end-t_start

            #Testing on various backtracking depths
            thread_time={p:100 for p in range(7)}
            thread_result={p:100 for p in range(7)}

            for run in range(3):
                for p,node in enumerate(back):

                    best=LB
  
                    t_start2 = timeit.default_timer()

                    solver = DLX(u + 1,rows_color,orig_row,D_color,LB,N_max,node)

                    if node>0:
                        for integer, obj in solver.search_range():
                            if obj > best:
                                best = obj
                                optimal = integer.copy()
                    t2 = timeit.default_timer() - t_start2
                    D_time = t+t2
                    if D_time<thread_time[p]:
                        thread_time[p]=D_time
                    thread_result[p] = best
                    

            results = Parallel(n_jobs=10,backend="loky")(delayed(run_single_i)(i,rows_color,D_color,orig_row,back,LB,N_max,u,t,thread_time,thread_result)
                               for i in range(thread))    

In [257]:

percentiles = [0.01,0.05, 0.25, 0.50, 0.75, 0.95, 0.99]#Backtracking depth percentile based on the beam load distribution
thread = 10# using 10 threads
dfs_best_of_fastest = []
dfs_fastest_of_best = []
N_max=60
n_colour=4
R=range(n_colour)
Gurobi_SOL=pd.DataFrame(columns=['Instance','User','Simulation','Solver','Computation Time(s)','Total Allocated Demand'])
OR_TOOLS_SOL=pd.DataFrame(columns=['Instance','User','Simulation','Solver','Computation Time(s)','Total Allocated Demand'])
for z in [4]:#4:Realistic, 8:Random, 12:Clustered
    for INS in [500]:#Number of users
        Data=pd.read_excel('Data/User Data/Instance_'+str(z)+'.xlsx')
        data=Data.iloc[:INS].copy()
        for h in range(20): #problems     
            beams=pd.read_csv('Data/Beam Candidate/Beams_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.csv',index_col=0)
            f = open('Data/Beam Candidate/Allocate_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.txt','r')
            allocate_data=f.read()
            f.close()
            allocate=eval(allocate_data)
            pairs_file = os.path.join('Data/Beam Candidate/', 'Pairs_'+str(z)+'_'+str(INS)+'_sample'+str(h)+'.json')
            with open(pairs_file, 'r') as f:
                pairs = json.load(f)

            rows_color, orig_row, pos, D_color,u,adj,cum_demands,back=input_structure(beams,allocate,data,pairs,percentiles)

            t_start = timeit.default_timer()

            #LB, greedy_sol= greedy_solution(rows_color, orig_row, pos, D_color, N_max, adj)

            #t_end = timeit.default_timer()
            #t=0

            #Testing on various backtracking depths
            
       

            results = Parallel(n_jobs=thread,backend="loky")(delayed(run_single_i)(i,rows_color,D_color,orig_row,back,0,N_max,u,t)
                               for i in range(thread)) 

            t_start = timeit.default_timer()
            
            best_of_fastest, fastest_of_best = cumulative_analysis(thread-1, results, range(len(back)))
            # ---------- Best of Fastest ----------
            dfs_best_of_fastest.append({
                "Instance": z,
                "User": INS,
                "Simulation": h,
                "Backtracking based on percentiles": best_of_fastest["p"],
                "Total Allocated Demand": best_of_fastest["value"],
                "Computation Time(s)": best_of_fastest["time"],
            })

            # ---------- Fastest of Best ----------
            dfs_fastest_of_best.append({
                "Instance": z,
                "User": INS,
                "Simulation": h,
                "Backtracking percentiles idx": fastest_of_best["p"],
                "Total Allocated Demand": fastest_of_best["value"],
                "Computation Time(s)": fastest_of_best["time"],               
            })

            print(f"Completed: z={z}, INS={INS}, simulation={h}")


# ---------------- Convert to DataFrames ----------------

DLX_v1 = pd.DataFrame(dfs_fastest_of_best)
DLX_v2 = pd.DataFrame(dfs_best_of_fastest)
DLX_v1.to_csv('DLXv1_results1.csv')
DLX_v2.to_csv('DLXv2_results1.csv')

Completed: z=4, INS=500, simulation=0
Completed: z=4, INS=500, simulation=1
Completed: z=4, INS=500, simulation=2
Completed: z=4, INS=500, simulation=3
Completed: z=4, INS=500, simulation=4
Completed: z=4, INS=500, simulation=5
Completed: z=4, INS=500, simulation=6
Completed: z=4, INS=500, simulation=7
Completed: z=4, INS=500, simulation=8
Completed: z=4, INS=500, simulation=9
Completed: z=4, INS=500, simulation=10
Completed: z=4, INS=500, simulation=11
Completed: z=4, INS=500, simulation=12
Completed: z=4, INS=500, simulation=13
Completed: z=4, INS=500, simulation=14
Completed: z=4, INS=500, simulation=15
Completed: z=4, INS=500, simulation=16
Completed: z=4, INS=500, simulation=17
Completed: z=4, INS=500, simulation=18
Completed: z=4, INS=500, simulation=19


In [82]:
percentiles = [0.01,0.05, 0.25, 0.50, 0.75, 0.95, 0.99]#Backtracking depth percentile based on the beam load distribution
thread = 10# using 10 threads
dfs_best_of_fastest = []
dfs_fastest_of_best = []
N_max=60
n_colour=4
R=range(n_colour)
Gurobi_SOL=pd.DataFrame(columns=['Instance','User','Simulation','Solver','Computation Time(s)','Total Allocated Demand'])
OR_TOOLS_SOL=pd.DataFrame(columns=['Instance','User','Simulation','Solver','Computation Time(s)','Total Allocated Demand'])
for z in [12]:#4:Realistic, 8:Random, 12:Clustered
    for INS in [5000]:#Number of users
        Data=pd.read_excel('Data/User Data/Instance_'+str(z)+'.xlsx')
        data=Data.iloc[:INS].copy()
        for h in range(1): #problems     
            beams=pd.read_csv('Data/Beam Candidate/Beams_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.csv',index_col=0)
            f = open('Data/Beam Candidate/Allocate_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.txt','r')
            allocate_data=f.read()
            f.close()
            allocate=eval(allocate_data)
            pairs_file = os.path.join('Data/Beam Candidate/', 'Pairs_'+str(z)+'_'+str(INS)+'_sample'+str(h)+'.json')
            with open(pairs_file, 'r') as f:
                pairs = json.load(f)

            rows_color, orig_row, pos, D_color,u,adj,cum_demands,back=input_structure(beams,allocate,data,pairs,percentiles)

            t_start = timeit.default_timer()

            LB, greedy_sol= greedy_solution(rows_color, orig_row, pos, D_color, N_max, adj)     

            results = Parallel(n_jobs=thread,backend="loky")(delayed(run_single_i)(i,rows_color,D_color,orig_row,back,LB,greedy_sol,N_max,u)
                               for i in range(thread)) 

            t_end = timeit.default_timer()
            print(t_end-t_start)
            

9.474935916000504


In [93]:
            t_start = timeit.default_timer()

            LB, greedy_sol= greedy_solution(rows_color, orig_row, pos, D_color, N_max, adj)     

            results = Parallel(n_jobs=thread,backend="loky")(delayed(run_single_i)(i,rows_color,D_color,orig_row,back,LB,greedy_sol,N_max,u,n_colour)
                               for i in range(thread)) 

            t_end = timeit.default_timer()
            print(t_end-t_start)

            best_obj, best_sol, best_depth = max(results, key=lambda x: x[0])

            print(best_obj)
            print(best_sol)
            print(best_depth)

8.896577708000223


In [95]:
best_obj, best_sol, best_depth = max(results, key=lambda x: x[0])

print(best_obj)
print(best_sol)
print(best_depth)

6823846.490000001
[8, 13, 16, 24, 36, 49, 54, 94, 133, 136, 178, 305, 381, 394, 431, 451, 534, 559, 731, 759, 815, 1018, 1064, 1158, 1161, 1273, 1342, 1427, 1477, 1593, 1660, 1692, 1702, 1704, 1742, 1785, 1857, 1884]
1


In [83]:
LB

5832122.350000001

In [84]:
results

[(6805838.940000001,
  [0,
   8,
   13,
   24,
   36,
   49,
   54,
   94,
   133,
   136,
   178,
   381,
   409,
   451,
   494,
   534,
   555,
   559,
   731,
   759,
   815,
   1018,
   1064,
   1158,
   1161,
   1273,
   1343,
   1366,
   1477,
   1660,
   1692,
   1702,
   1704,
   1742,
   1785,
   1829,
   1857],
  1),
 (6668485.500000001,
  [4,
   8,
   13,
   24,
   36,
   49,
   54,
   94,
   133,
   136,
   178,
   245,
   262,
   381,
   451,
   534,
   559,
   691,
   731,
   759,
   815,
   1018,
   1064,
   1158,
   1161,
   1273,
   1340,
   1477,
   1660,
   1692,
   1702,
   1704,
   1742,
   1749],
  1),
 (6823846.490000001,
  [8,
   13,
   16,
   24,
   36,
   49,
   54,
   94,
   133,
   136,
   178,
   305,
   381,
   394,
   431,
   451,
   534,
   559,
   731,
   759,
   815,
   1018,
   1064,
   1158,
   1161,
   1273,
   1342,
   1427,
   1477,
   1593,
   1660,
   1692,
   1702,
   1704,
   1742,
   1785,
   1857,
   1884],
  1),
 (6789793.330000003,
  [12,

In [258]:
import sys

sys.path.insert(
    0,
    "/Users/ryanli/Documents/Codex/2026-06-10/"
    "files-mentioned-by-the-user-functions1/outputs"
)

from dlx_experiment import run

v1, v2 = run(
    data_dir="/Users/ryanli/Documents/GitHub/DLX_plus/Data",
    output_dir="/Users/ryanli/Documents/GitHub/DLX_plus",
    sample_count=20,
)

print(v1, v2)

Completed: z=4, INS=500, simulation=0
Completed: z=4, INS=500, simulation=1
Completed: z=4, INS=500, simulation=2
Completed: z=4, INS=500, simulation=3
Completed: z=4, INS=500, simulation=4
Completed: z=4, INS=500, simulation=5
Completed: z=4, INS=500, simulation=6
Completed: z=4, INS=500, simulation=7
Completed: z=4, INS=500, simulation=8
Completed: z=4, INS=500, simulation=9
Completed: z=4, INS=500, simulation=10
Completed: z=4, INS=500, simulation=11
Completed: z=4, INS=500, simulation=12
Completed: z=4, INS=500, simulation=13
Completed: z=4, INS=500, simulation=14
Completed: z=4, INS=500, simulation=15
Completed: z=4, INS=500, simulation=16
Completed: z=4, INS=500, simulation=17
Completed: z=4, INS=500, simulation=18
Completed: z=4, INS=500, simulation=19
/Users/ryanli/Documents/GitHub/DLX_plus/DLXv1_results1.csv /Users/ryanli/Documents/GitHub/DLX_plus/DLXv2_results1.csv


In [2]:
import sys

sys.path.insert(
    0,
    "/Users/ryanli/Documents/Codex/2026-06-10/"
    "files-mentioned-by-the-user-functions1/outputs"
)

from dlx_experiment import run_all

v1, v2 = run_all(
    data_dir="/Users/ryanli/Documents/GitHub/DLX_plus/Data",
    output_dir="/Users/ryanli/Documents/GitHub/DLX_plus",
    sample_count=20,
)

print(v1, v2)

Completed: z=4, INS=500, simulation=0
Completed: z=4, INS=500, simulation=1
Completed: z=4, INS=500, simulation=2
Completed: z=4, INS=500, simulation=3
Completed: z=4, INS=500, simulation=4
Completed: z=4, INS=500, simulation=5
Completed: z=4, INS=500, simulation=6
Completed: z=4, INS=500, simulation=7
Completed: z=4, INS=500, simulation=8
Completed: z=4, INS=500, simulation=9
Completed: z=4, INS=500, simulation=10
Completed: z=4, INS=500, simulation=11
Completed: z=4, INS=500, simulation=12
Completed: z=4, INS=500, simulation=13
Completed: z=4, INS=500, simulation=14
Completed: z=4, INS=500, simulation=15
Completed: z=4, INS=500, simulation=16
Completed: z=4, INS=500, simulation=17
Completed: z=4, INS=500, simulation=18
Completed: z=4, INS=500, simulation=19
Completed: z=4, INS=1000, simulation=0
Completed: z=4, INS=1000, simulation=1
Completed: z=4, INS=1000, simulation=2
Completed: z=4, INS=1000, simulation=3
Completed: z=4, INS=1000, simulation=4
Completed: z=4, INS=1000, simulatio

In [2]:
import sys

sys.path.insert(
    0,
    "/Users/ryanli/Documents/Codex/2026-06-10/"
    "files-mentioned-by-the-user-functions1/outputs"
)

In [3]:
from dlx_experiment import run_all

results = run_all(
    data_dir="/Users/ryanli/Documents/GitHub/DLX_plus/Data",
    output_dir="/Users/ryanli/Documents/GitHub/DLX_plus",
    instances=(4, 8, 12),
    users=(500, 1000, 2500, 5000),
    samples=range(20),
    R=range(4),
    n_colour=4,
)

Completed: z=4, INS=500, simulation=0
Completed: z=4, INS=500, simulation=1
Completed: z=4, INS=500, simulation=2
Completed: z=4, INS=500, simulation=3
Completed: z=4, INS=500, simulation=4
Completed: z=4, INS=500, simulation=5
Completed: z=4, INS=500, simulation=6
Completed: z=4, INS=500, simulation=7
Completed: z=4, INS=500, simulation=8
Completed: z=4, INS=500, simulation=9
Completed: z=4, INS=500, simulation=10
Completed: z=4, INS=500, simulation=11
Completed: z=4, INS=500, simulation=12
Completed: z=4, INS=500, simulation=13
Completed: z=4, INS=500, simulation=14
Completed: z=4, INS=500, simulation=15
Completed: z=4, INS=500, simulation=16
Completed: z=4, INS=500, simulation=17
Completed: z=4, INS=500, simulation=18
Completed: z=4, INS=500, simulation=19
Completed: z=4, INS=1000, simulation=0
Completed: z=4, INS=1000, simulation=1
Completed: z=4, INS=1000, simulation=2
Completed: z=4, INS=1000, simulation=3
Completed: z=4, INS=1000, simulation=4
Completed: z=4, INS=1000, simulatio

In [4]:
results

,Instance,User,Simulation,R,n_colour,integer,objective,time
0,4,500,0,0 1 2 3,4,"[0, 8, 28, 40, 44, 52, 65, 77, 124, 128, 133, ...",962750.00,0.001873
1,4,500,1,0 1 2 3,4,"[0, 16, 40, 44, 64, 85, 89, 105, 160, 168, 181...",965875.00,0.003063
2,4,500,2,0 1 2 3,4,"[0, 4, 24, 28, 40, 61, 65, 81, 128, 132, 141, ...",971325.00,0.002002
3,4,500,3,0 1 2 3,4,"[0, 4, 12, 16, 28, 133, 137, 141, 144, 148, 15...",963325.00,0.002669
4,4,500,4,0 1 2 3,4,"[0, 12, 24, 44, 48, 57, 73, 77, 113, 120, 124,...",953100.00,0.002291
...,...,...,...,...,...,...,...,...
235,12,5000,15,0 1 2 3,4,"[12, 17, 20, 48, 52, 81, 98, 157, 164, 245, 30...",6823216.77,0.046978
236,12,5000,16,0 1 2 3,4,"[4, 44, 53, 64, 68, 89, 118, 121, 133, 204, 27...",6980823.21,0.058104
237,12,5000,17,0 1 2 3,4,"[4, 9, 12, 20, 52, 109, 129, 136, 186, 193, 27...",6858499.52,0.061890
238,12,5000,18,0 1 2 3,4,"[4, 8, 16, 33, 36, 41, 102, 140, 153, 157, 226...",6738620.34,0.044315


In [247]:
DLX_v1 = pd.DataFrame(dfs_fastest_of_best)
DLX_v2 = pd.DataFrame(dfs_best_of_fastest)
DLX_v1.to_csv('DLXv1_results1.csv')
DLX_v2.to_csv('DLXv2_results1.csv')

In [135]:
def run_ML_i(i, rows_color, D_color, orig_row, back, LB, N_max, u, t,n_colour=4):
    """
    Run DLX+ for one starting index i across all percentile settings.

    For each percentile, the function:
    - sets the starting row position,
    - adjusts the backtracking depth,
    - runs DLX+ if the depth is positive,
    - records the total time,
    - records the best objective value found.
    """
    
    t_start2 = timeit.default_timer()
    # Each vertex contributes four colour rows.
    begin = i * n_colour

    # Adjust the remaining backtracking depth.
    node = back- i * n_colour

    # Initialise the best objective value.
    best = LB

    # Run DLX+ only if there are remaining nodes to explore.
    if node > 0:
        solver = DLX(
                u + 1,
                rows_color[begin:],
                orig_row[begin:],
                D_color[begin:],
                LB,
                N_max,
                node
            )

        for integer, obj in solver.search_range():
            if obj > best:
                best = obj
                optimal = integer.copy()

    t2 = timeit.default_timer() - t_start2


    return t + t2, best

In [162]:
### DLX+:ML
from lightgbm import LGBMClassifier

###Obtain Features of Each Problem
n_colour=4
test_rows = []
for z in [4,8,12]:#4:Realistic, 8:Random, 12:Clustered
    for INS in [500,1000,2500,5000]:#Number of users
        Data=pd.read_excel('Data/User Data/Instance_'+str(z)+'.xlsx')
        data=Data.iloc[:INS].copy()
         
        for h in range(20):      
            back=DLX_v1[(DLX_v1["Instance"] == z) & (DLX_v1["User"] == INS)& (DLX_v1["Simulation"] == h)]['Backtracking percentiles idx'].values[0]
             
            beams=pd.read_csv('Data/Beam Candidate/Beams_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.csv',index_col=0)
            f = open('Data/Beam Candidate/Allocate_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.txt','r')
            allocate_data=f.read()
            f.close()
            allocate=eval(allocate_data)
            pairs_file = os.path.join('Data/Beam Candidate/', 'Pairs_'+str(z)+'_'+str(INS)+'_sample'+str(h)+'.json')
            with open(pairs_file, 'r') as f:
                pairs = json.load(f)
            
            rows_color, D_color,user_conlict,num_cons,adj,pairs,max_cliques=generate_features(beams,allocate,data,pairs)

            u=max([j for i in rows_color for j in i])
            X = extract_features_FBO(rows_color,D_color,N_max,len(beams),user_conlict,max_cliques,adj,num_cons,pairs,n_colour)
            test_rows.append({"Instance": z,"User": INS,"Simulation": h,**X,"Backtracking percentiles idx": back}) 
df_ML = pd.DataFrame(test_rows)

percentiles = [0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]#Backtracking limits were selected to reflect practical operating points, ranging from minimal backtracking (1) to aggressive partial backtracking (60)
thread=10
N_max=60
DLX_ML_sol=pd.DataFrame(columns=['Instance','User','Simulation','Computation Time(s)','Total Allocated Demand','Backtracking based on percentiles'])
FEATURES = ['B', 'U', 'num_rows', 'density', 'avg_degree', 'max_degree', 'degree_std', 'D_mean', 'D_std', 'D_cv',
 'avg_row_len', 'max_row_len', 'row_density'
]

model = LGBMClassifier(objective="multiclass",num_class=7,
                       n_estimators=300,max_depth=6,learning_rate=0.05,subsample=0.85,colsample_bytree=0.85,
                       min_child_samples=5,
                       class_weight="balanced",
                       verbose=-1)
        
for z in [4,8,12]:
    for INS in [500,1000,2500,5000]:
        Data=pd.read_excel('Data/User Data/Instance_'+str(z)+'.xlsx')
        data=Data.iloc[:INS].copy()
        
        for h in range(20):      


            # Create boolean mask for test
            mask_test = (
                (df_ML['Instance'] == z) &
                (df_ML['User'] == INS) &
                (df_ML['Simulation'] == h)
            )

            # Split into test and train
            df_test = df_ML.loc[mask_test].reset_index(drop=True)
            df_train = df_ML.loc[~mask_test].reset_index(drop=True)
            
            X_train = df_train[FEATURES]
            y_train = df_train["Backtracking percentiles idx"]
            model.fit(X_train, y_train)
            
             
            beams=pd.read_csv('Data/Beam Candidate/Beams_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.csv',index_col=0)
            f = open('Data/Beam Candidate/Allocate_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.txt','r')
            allocate_data=f.read()
            f.close()
            allocate=eval(allocate_data)
            pairs_file = os.path.join('Data/Beam Candidate/', 'Pairs_'+str(z)+'_'+str(INS)+'_sample'+str(h)+'.json')
            with open(pairs_file, 'r') as f:
                pairs = json.load(f)

            rows_color, orig_row, pos, D_color,u,adj,cum_demands,back=input_structure(beams,allocate,data,pairs,percentiles)

            t_start = timeit.default_timer()

            LB, greedy_sol= greedy_solution(rows_color, orig_row, pos, D_color, N_max, adj)
            
            p = model.predict(df_test[FEATURES])[0]

            target = percentiles[p] * LB
            back = np.argmin(np.abs(cum_demands - target)) + 1

            

            results = Parallel(n_jobs=thread,backend="loky")(delayed(run_ML_i)(i,rows_color,D_color,orig_row,back,LB,N_max,u,t_end)
                               for i in range(thread))   

            t_end = timeit.default_timer()-t_start
            max_time = max(t for t, v in results)
            best_value = max(v for t, v in results)
            DLX_ML_sol.loc[len(DLX_ML_sol)]=[z,INS,h,max_time,best_value,percentiles[p]]

DLX_ML_sol.to_csv('DLX_ML_results.csv')

In [10]:
percentiles = [0.01,0.05, 0.25, 0.50, 0.75, 0.95, 0.99]#Backtracking limits were selected to reflect practical operating points, ranging from minimal backtracking (1) to aggressive partial backtracking (60)
thread = 10# using 10 threads

DLX_LP_sol=pd.DataFrame(columns=['Instance','User','Simulation','Computation Time(s)','Total Allocated Demand','Backtracking percentiles idx'])

N_max=60
R=range(4)
for z in [4,8,12]:
    for INS in [500,1000,2500,5000]:
        Data=pd.read_excel('Data/User Data/Instance_'+str(z)+'.xlsx')
        data=Data.iloc[:INS].copy()
        
        for h in range(20):        
            back=DLX_v1[(DLX_v1["Instance"] == z) & (DLX_v1["User"] == INS)& (DLX_v1["Simulation"] == h)]['Backtracking percentiles idx'].values[0]
             
            beams=pd.read_csv('Data/Beam Candidate/Beams_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.csv',index_col=0)
            f = open('Data/Beam Candidate/Allocate_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.txt','r')
            allocate_data=f.read()
            f.close()
            allocate=eval(allocate_data)
            pairs_file = os.path.join('Data/Beam Candidate/', 'Pairs_'+str(z)+'_'+str(INS)+'_sample'+str(h)+'.json')
            with open(pairs_file, 'r') as f:
                pairs = json.load(f)
            

            rows=list(allocate.values())

            B=len(beams)

            D=beams.Demand

            Q_b = {i: [] for i in data.index}
            for beam, users in allocate.items():
                for u_ in users:
                    Q_b[u_].append(beam)

            C=removeSublist(list(Q_b.values())) 

            rows=incidence_transpose_fast(C)

            adj = defaultdict(set)
            for i, j in pairs:
                adj[i].add(j)
                adj[j].add(i)            

            G = nx.Graph()
            G.add_edges_from(pairs)
            max_cliques=list(nx.find_cliques(G))

            t_start = timeit.default_timer()

            best=0
            t=0
            t1=0


            integer4_int_color,integer4_partial_color,fraction4,int_D=gurobi_LP(beams,Q_b,N_max,max_cliques,R,'C')

            if integer4_partial_color or fraction4:
                
                B_set=s = set(range(B))
                if integer4_int_color:
                    int_set = set(integer4_int_color)
                    # Step 1: collect all users served by beams in integer4
                    users_served = set().union(*(allocate[b] for b in integer4_int_color))
                    # Step 2: collect all beams serving those users
                    to_remove = int_set | set().union(*(Q_b[u] for u in users_served))
                    B_set-=to_remove

                B_set_list = list(B_set)
                D_set = np.array([D[i] for i in B_set])
                sort_index = np.argsort(-D_set)  # descending order


                top_indices = []
                median_indices = []
                rest_indices = []
                for i in sort_index:
                    idx=B_set_list[i]
                    if idx in integer4_partial_color:
                        top_indices.append(i)
                    elif idx in fraction4:
                        median_indices.append(i)
                    else:
                        rest_indices.append(i)
                    
                sort_index_reordered = top_indices+median_indices+rest_indices

                D_sort_list = D_set[sort_index_reordered].tolist()

                cum_demands = list(itertools.accumulate(D_sort_list))
                total_demands=cum_demands[-1]
                cum_demands= np.array(cum_demands)
                
                rows_sort = [allocate[B_set_list[i]] for i in sort_index_reordered]

                all_cols = sorted(set(c for row in rows_sort for c in row))
                    
                col_mapping = {old: new for new, old in enumerate(all_cols)}

                reindexed_columns = [[col_mapping[c] for c in row] for row in rows_sort]

                row_mapping = {old: new for new, old in enumerate([B_set_list[i] for i in sort_index_reordered])}


                cliques_B_set = []
                for clique in max_cliques:
                    remapped = [row_mapping[b] for b in clique if b in row_mapping]
                    if len(remapped) > 1:
                        cliques_B_set.append(remapped)
                        
                t= timeit.default_timer()-t_start

                rows_color=[]
                D_color=[]
                orig_row = []
                for idx, row in enumerate(reindexed_columns):
                    d = D_sort_list[idx]
                    for j in range(4):
                        rows_color.append(row.copy())
                        D_color.append(d) 
                        orig_row.append([idx,j])

                u = len(all_cols)+1
                for clique in cliques_B_set:
                    idxs = [j * 4 for j in clique]
                    for k in range(4):
                        for base in idxs:
                            rows_color[base + k].append(u)
                        u += 1

                t_start1 = timeit.default_timer()

                if integer4_int_color:

                    connected = defaultdict(set)

                    for p, q in pairs:
                        if p in int_set and q in B_set:
                            connected[row_mapping[q]].add(integer4_int_color[p])
                        elif q in int_set and p in B_set:
                            connected[row_mapping[p]].add(integer4_int_color[q])

                    for i in sorted(list(connected), reverse=True):
                        for j in sorted(connected[i], reverse=True):
                            idx=i*4+j
                            del rows_color[idx]
                            del orig_row[idx]
                            del D_color[idx]
                            

                back=[]

                for p in percentiles:
                    target = p * total_demands
                    node = int(np.argmin(np.abs(cum_demands - target)))
                    back.append(node*4+1)

                thread_time={p:0 for p in range(7)}
                thread_result={p:0 for p in range(7)}

                t1= t+timeit.default_timer()-t_start

                capacity=N_max-len(integer4_int_color)

                results = Parallel(n_jobs=thread,backend="loky")(delayed(run_single_i)(i,rows_color,D_color,orig_row,back,best,capacity,u,t1)
                               for i in range(thread))    

                
                best_of_fastest, fastest_of_best = cumulative_analysis(thread-1, results, range(len(back)))
                DLX_LP_sol.loc[len(DLX_LP_sol)]=[z,INS,h,fastest_of_best["time"],fastest_of_best["value"]+int_D,fastest_of_best["p"]]


            else:
                
                best+=int_D
                t_end = timeit.default_timer()
                D_time=t_end-t_start
            
                DLX_LP_sol.loc[len(DLX_LP_sol)]=[z,INS,h,D_time,best,None]
            print(f"Completed: z={z}, INS={INS}, simulation={h}")



DLX_LP_sol.to_csv('DLX_LP_results.csv')


Completed: z=4, INS=500, simulation=0
Completed: z=4, INS=500, simulation=1
Completed: z=4, INS=500, simulation=2
Completed: z=4, INS=500, simulation=3
Completed: z=4, INS=500, simulation=4
Completed: z=4, INS=500, simulation=5
Completed: z=4, INS=500, simulation=6
Completed: z=4, INS=500, simulation=7
Completed: z=4, INS=500, simulation=8
Completed: z=4, INS=500, simulation=9
Completed: z=4, INS=500, simulation=10
Completed: z=4, INS=500, simulation=11
Completed: z=4, INS=500, simulation=12
Completed: z=4, INS=500, simulation=13
Completed: z=4, INS=500, simulation=14
Completed: z=4, INS=500, simulation=15
Completed: z=4, INS=500, simulation=16
Completed: z=4, INS=500, simulation=17
Completed: z=4, INS=500, simulation=18
Completed: z=4, INS=500, simulation=19


In [136]:
def gurobi_LP(rows_color,D_color,N,Cons):

    V=range(len(rows_color))

    model = gp.Model("MIP")

    model.setParam('TimeLimit', 3600)

    x = model.addVars(V, vtype='C',lb=0,ub=1, name="x")

    # SPK constraints
    model.addConstrs(gp.quicksum(x[b] for b in Cons[p]) <= 1 for p in range(len(Cons)))

    # Cardinality constraint
    model.addConstr(gp.quicksum(x[b] for b in range(len(rows_color))) <= N)

    # Objective
    model.setObjective(gp.quicksum(D_color[b] * x[b] for b in range(len(rows_color))),gp.GRB.MAXIMIZE)

    model.optimize()
    integer=[]
    fraction = []
    int_D=0
    for b in V:
        val = round(x[b].x,2)
        if val > 0.99:
            integer.append(b)
            int_D+=D_color[b]  
        elif val > 1e-6:           # true fractional
            fraction.append(b)
    
    return integer,fraction,int_D,model.objVal,model.Runtime

In [160]:
percentiles = [0.01,0.05, 0.25, 0.50, 0.75, 0.95, 0.99]#Backtracking limits were selected to reflect practical operating points, ranging from minimal backtracking (1) to aggressive partial backtracking (60)
thread = 10# using 10 threads

DLX_LP_sol=pd.DataFrame(columns=['Instance','User','Simulation','Computation Time(s)','Total Allocated Demand','Backtracking percentiles idx'])

N_max=60
R=range(n_colour)
for z in [4,8,12]:
    for INS in [500,1000,2500,5000]:
        Data=pd.read_excel('Data/User Data/Instance_'+str(z)+'.xlsx')
        data=Data.iloc[:INS].copy()
        
        for h in range(20):        
             
            beams=pd.read_csv('Data/Beam Candidate/Beams_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.csv',index_col=0)
            f = open('Data/Beam Candidate/Allocate_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.txt','r')
            allocate_data=f.read()
            f.close()
            allocate=eval(allocate_data)
            pairs_file = os.path.join('Data/Beam Candidate/', 'Pairs_'+str(z)+'_'+str(INS)+'_sample'+str(h)+'.json')
            with open(pairs_file, 'r') as f:
                pairs = json.load(f)
            

            rows_color, orig_row, pos, D_color,u,adj,cum_demands,back=input_structure(beams,allocate,data,pairs,percentiles)

            Cons=incidence_transpose_fast(rows_color)

            t_start = timeit.default_timer()

            integer,fraction,int_D,obj,run_time=gurobi_LP(rows_color,D_color,N_max,Cons)

            best=0
            t1=0

            if fraction:
                solver = DLX(u + 1,rows_color,orig_row,D_color,0,N_max,0)
                for i in integer:
                    row_header = solver.rows[i]
                    first_col=row_header.R
                    solver.row_loop(first_col)
                remaining_nodes=[]
                Root=solver.root
                Node=Root.D
                while Node!=Root:
                    remaining_nodes.append(Node.name)
                    Node=Node.D
                remaining_rows=[]
                remaining_demand=[]
                remaining_orig_row=[]
                demand_structure=[]
                last_row=-1
                num_row=1
                pos={}
                len_demand_structure=0
                len_remaining_rows=0
                for row in remaining_nodes:
                    remaining_rows.append(rows_color[row])
                    remaining_demand.append(D_color[row])
                    remaining_orig_row.append(orig_row[row])
                    if orig_row[row][0]!=last_row:
                        demand_structure.append(D_color[row])
                        pos[len_demand_structure]=len_remaining_rows
                        len_demand_structure+=1
                    last_row=orig_row[row][0]
                    len_remaining_rows+=1
                cum_demands = list(itertools.accumulate(demand_structure))
                total_demands=cum_demands[-1]
                cum_demands= np.array(cum_demands)
                back=[]

                for p in percentiles:#p backtracking depth
                    target = p * total_demands
                    node = int(np.argmin(np.abs(cum_demands - target)))
                    back.append(pos[node]+1)                    

               

                capacity=N_max-len(integer)

                t_end = timeit.default_timer()
                t=t_end-t_start

                results = Parallel(n_jobs=thread,backend="loky")(delayed(run_single_i)(i,remaining_rows,remaining_demand,remaining_orig_row,back,best,capacity,u,t)
                               for i in range(thread))    

                
                best_of_fastest, fastest_of_best = cumulative_analysis(thread-1, results, range(len(back)))
                DLX_LP_sol.loc[len(DLX_LP_sol)]=[z,INS,h,fastest_of_best["time"],fastest_of_best["value"]+int_D,fastest_of_best["p"]]


            else:
                
                t_end = timeit.default_timer()
                D_time=t_end-t_start
            
                DLX_LP_sol.loc[len(DLX_LP_sol)]=[z,INS,h,D_time,int_D,None]
            print(f"Completed: z={z}, INS={INS}, simulation={h}")



DLX_LP_sol.to_csv('DLX_LP_results.csv')


Set parameter TimeLimit to value 3600
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (mac64[rosetta2] - Darwin 23.6.0 23G93)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 1853 rows, 1188 columns and 14996 nonzeros
Model fingerprint: 0xc2e2e9c8
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e+02, 1e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 6e+01]
Presolve removed 320 rows and 53 columns
Presolve time: 0.01s
Presolved: 1533 rows, 1135 columns, 14873 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    1.7426010e+06   9.894750e+01   0.000000e+00      0s
     126    9.7987500e+05   0.000000e+00   0.000000e+00      0s

Use crossover to convert LP symmetric solution to basic solution...
Crossover log...

       0 DPushes remaining with DInf 0.0000000e+00                 0s

     177 PPushes remaining with PInf 0.0000000e+0

### Time_Limit

In [ ]:
def gurobi_SPK(rows_color, D_color, N, Cons, limit):

    V = range(len(rows_color))

    model = gp.Model("MIP")
    model.setParam("TimeLimit", limit)

    x = model.addVars(V, vtype=gp.GRB.BINARY, name="x")

    model.addConstrs(
        gp.quicksum(x[b] for b in Cons[p]) <= 1
        for p in range(len(Cons))
    )

    model.addConstr(gp.quicksum(x[b] for b in V) <= N)

    model.setObjective(
        gp.quicksum(D_color[b] * x[b] for b in V),
        gp.GRB.MAXIMIZE
    )

    model.optimize()

    # No feasible solution found
    if model.SolCount == 0:
        return [], None, model.Runtime

    integer = [v for v in V if x[v].X > 0.5]

    if model.Status == gp.GRB.OPTIMAL:
        status = "Optimal"
    else:
        status = "Feasible"

    return integer, model.ObjVal, model.Runtime

def cpsat_SPK(rows_color,D_color,N,Cons,limit,log=True):
    
    V=range(len(rows_color))

    model = cp_model.CpModel()

    # Variables
    x = {b: model.NewBoolVar(f"x[{b}]") for b in V}
   

    # User coverage constraints
    for q_set in range(len(Cons)):
        model.Add(sum(x[b] for b in Cons[q_set]) <= 1)

    # Cardinality constraint
    model.Add(sum(x[b] for b in V) <= N)

    # Objective
    model.Maximize(sum(int(D_color[b]) * x[b] for b in V))

    # Solver
    solver = cp_model.CpSolver()
    solver.parameters.max_time_in_seconds = limit
    #solver.parameters.num_search_workers = 8
    #solver.parameters.log_search_progress = log

    start = time.time()
    status = solver.Solve(model)
    runtime = time.time() - start

    if status in [cp_model.OPTIMAL, cp_model.FEASIBLE]:

        integer = [b for b in V if solver.Value(x[b]) >0.99]
        obj=solver.ObjectiveValue()
    else:
        integer = []
        obj = None
 

    return integer, obj, runtime

In [245]:
N_max=60
n_colour=4
R=range(n_colour)
Gurobi_SOL=pd.DataFrame(columns=['Instance','User','Simulation','Solver','Computation Time(s)','Total Allocated Demand'])
OR_TOOLS_SOL=pd.DataFrame(columns=['Instance','User','Simulation','Solver','Computation Time(s)','Total Allocated Demand'])
DLX_best = pd.read_csv('/Users/ryanli/Documents/GitHub/DLX_plus/DLXv1_results.csv')
for z in [4,8,12]:#4:Realistic, 8:Random, 12:Clustered
    for INS in [500,1000,2500,5000]:#Number of users
        Data=pd.read_excel('Data/User Data/Instance_'+str(z)+'.xlsx')
        data=Data.iloc[:INS].copy()
        for h in range(20): #problems     
            beams=pd.read_csv('Data/Beam Candidate/Beams_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.csv',index_col=0)
            f = open('Data/Beam Candidate/Allocate_'+str(z)+'_'+str(INS)+'_60_4_sample'+str(h)+'.txt','r')
            allocate_data=f.read()
            f.close()
            allocate=eval(allocate_data)
            pairs_file = os.path.join('Data/Beam Candidate/', 'Pairs_'+str(z)+'_'+str(INS)+'_sample'+str(h)+'.json')
            with open(pairs_file, 'r') as f:
                pairs = json.load(f)

            D_time=DLX_best[(DLX_best['Instance'] == z) & (DLX_best['User'] == INS)&(DLX_best['Simulation'] == h)]['Computation Time(s)'].values[0]

            rows_color, orig_row, pos, D_color,u,adj,cum_demands,back=input_structure(beams,allocate,data,pairs,percentiles)

            Cons=incidence_transpose_fast(rows_color)

            t_start = timeit.default_timer()

            guro_integer,guro_obj,guro_solver_time=gurobi_SPK(rows_color,D_color,N_max,Cons,D_time)

            t_end = timeit.default_timer()

            Gurobi_SOL.loc[len(Gurobi_SOL)]=[z,INS,h,'Gurobi',t_end-t_start,guro_obj]

            t_start = timeit.default_timer()

            or_integer,or_obj,or_solver_time=cpsat_SPK(rows_color,D_color,N_max,Cons, D_time)

            t_end = timeit.default_timer()

            OR_TOOLS_SOL.loc[len(OR_TOOLS_SOL)]=[z,INS,h,'OR_Tools',t_end-t_start,or_obj]
            print(f"Completed: z={z}, INS={INS}, simulation={h}")


Gurobi_SOL.to_csv('Gurobi_results_timelimit.csv')
OR_TOOLS_SOL.to_csv('OR_TOOLS_results_timelimit.csv')

Set parameter TimeLimit to value 0.0158070410000021
Gurobi Optimizer version 11.0.2 build v11.0.2rc0 (mac64[rosetta2] - Darwin 23.6.0 23G93)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 1853 rows, 1188 columns and 14996 nonzeros
Model fingerprint: 0x46f38d97
Variable types: 0 continuous, 1188 integer (1188 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [4e+02, 1e+05]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 6e+01]
Found heuristic solution: objective 950350.00000
Presolve removed 803 rows and 106 columns
Presolve time: 0.01s

Explored 0 nodes (0 simplex iterations) in 0.02 seconds (0.02 work units)
Thread count was 1 (of 10 available processors)

Solution count 1: 950350 
No other solutions better than 0

Time limit reached
Best objective 9.503500000000e+05, best bound -, gap -
Completed: z=4, INS=500, simulation=0
Set parameter TimeLimit to value 0